# Dialect ASR experiment on Colab

Fine-tunes Whisper small on ViMD speech from many provinces and compares the word error rate (WER) before and after.

**Before you start:** Runtime > Change runtime type > a GPU (T4 is enough).
ViMD is CC BY-NC-ND 4.0: use it for coursework and research only.

In [ ]:
!nvidia-smi -L
!df -h /content | tail -1   # free disk matters: see README "Disk is the limit"

## 1. Get the code and install

Use the branch that holds this folder. If the repository is private, upload the folder to Drive instead.

In [ ]:
BRANCH = "feat/smart-import"
REPO = "https://github.com/anvuhoang/seafood-marketplace-web.git"
!git clone --depth 1 --branch {BRANCH} {REPO} repo
%cd repo/ai-service/speech/finetune
!pip install -q -r requirements.txt

## 2. Choose provinces

Values: names without accents (`QuangNgai,KhanhHoa`), `coastal` (28 provinces), `region:Central`, `region:South`, `region:North`, `all`. They can be mixed.

- **First run: keep the 3-province smoke test** so a mistake shows up before a big download.
- **Real run:** train on a group that fits the disk, and evaluate on a wider one (test files are small).

In [ ]:
TRAIN_PROVINCES = "QuangNgai,KhanhHoa,BaRiaVungTau"   # smoke test; then e.g. "region:Central"
EVAL_PROVINCES = TRAIN_PROVINCES                       # then e.g. "coastal"
!python plan_download.py --provinces {TRAIN_PROVINCES}
!python plan_download.py --provinces {EVAL_PROVINCES}

## 3. Baseline: the model before fine-tuning

Run this first so there is something to compare with.

In [ ]:
!python eval_wer.py --model openai/whisper-small --provinces {EVAL_PROVINCES} --name baseline-whisper-small

Optional: the model the app uses today (Groq). Needs a free key from console.groq.com. Paste it when asked; **do not save the notebook with the key in it**. `--limit 5` keeps it to a few minutes.

In [ ]:
# import os, getpass; os.environ["GROQ_API_KEY"] = getpass.getpass("Groq key: ")
# !python eval_wer.py --model groq:whisper-large-v3-turbo --provinces {EVAL_PROVINCES} --limit 5 --name baseline-groq-turbo

## 4. Fine-tune

About 1000 steps takes roughly an hour on a T4 (an estimate, not measured). Lower `--max-steps` for a first try.

In [ ]:
!python finetune.py --train-provinces {TRAIN_PROVINCES} --max-steps 1000

## 5. Evaluate the fine-tuned model on the same test clips

In [ ]:
!python eval_wer.py --model outputs/whisper-small-vimd --provinces {EVAL_PROVINCES} --name finetuned-whisper-small

## 6. Compare and keep the results

The JSON files in `results/` hold the numbers. Download them (the model itself is large; save it to Drive only if you need it).

In [ ]:
import json, glob
for path in sorted(glob.glob("results/*-test.json")):
    r = json.load(open(path))
    print(f"{r['model']:<45} overall WER {r['overall_wer']:.1%}  ({r['clips']} clips)")
    for region, v in r["by_region"].items(): print(f"    region {region:<9}{v['wer']:.1%}  ({v['clips']} clips)")

In [ ]:
from google.colab import files
!zip -q -r results.zip results
files.download("results.zip")